In [ ]:
# Environment
import os, sys
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)
while not os.path.exists('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print(f"\U0001f4c1 Working: {os.getcwd()}")

# 10 - BigQuery GDELT: Expand Buildout Promise Labels

**Goal**: Mine GDELT (Global Database of Events, Language, and Tone) on BigQuery to discover new data center buildout announcements beyond the current 34 labeled promises.

**Current bottleneck**: Only 34 buildout promises with `promise_kept` labels across 20 tickers. This notebook queries GDELT v2 (~250M events from 50K+ news sources) to find candidate announcements from hyperscalers and colo providers.

**Approach**:
- **GKG** (Global Knowledge Graph): filter articles by organization names + infrastructure/investment themes
- **Events**: filter by Actor1 (company) + EventCode (economic cooperation/investment) + ActionGeo (location)
- Parse results into structured candidate buildout records

**Output**: `buildout_candidates_bigquery.csv` — new candidate announcements for manual validation

## 1. Setup & Auth

### On Colab
Run `colab auth` to authenticate with your GCP project:
```
colab auth -s <session>
```
This uses Application Default Credentials. The project must have BigQuery API enabled.

Set your project ID:
```
os.environ['GOOGLE_CLOUD_PROJECT'] = 'your-project-id'
```

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
from datetime import datetime, timedelta
from pathlib import Path
warnings.filterwarnings('ignore')

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

from google.cloud import bigquery

# Set your GCP project
PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", "project-21db66e7-39ca-4fda-b4e")
client = bigquery.Client(project=PROJECT)
print(f"\u2705 BQ client ready (project={PROJECT})")

# Load existing promises for dedup
promises_path = Path('data/raw/buildout_promises_expanded.csv')
if promises_path.exists():
    df_existing = pd.read_csv(promises_path)
    print(f"Existing promises: {len(df_existing)}")
else:
    df_existing = pd.DataFrame()
    print("No existing promises found")

## 2. Define Target Companies

These are the hyperscalers and colo providers whose buildout announcements we want to track.

In [ ]:
# Target companies for buildout announcements
TARGET_COMPANIES = [
    # Hyperscalers
    ('Microsoft', ['Microsoft', 'MSFT']),
    ('Google', ['Google', 'Alphabet', 'GOOGL', 'GOOG']),
    ('Amazon', ['Amazon', 'AWS', 'AMZN']),
    ('Meta', ['Meta', 'Facebook', 'META']),
    ('Oracle', ['Oracle', 'ORCL']),
    # Colo / digital infra
    ('Equinix', ['Equinix', 'EQIX']),
    ('Digital Realty', ['Digital Realty', 'DLR']),
    ('American Tower', ['American Tower', 'AMT']),
    # GPU cloud
    ('CoreWeave', ['CoreWeave']),
    ('Crusoe', ['Crusoe Energy']),
    ('Lambda', ['Lambda Labs']),
]

print(f"Tracking {len(TARGET_COMPANIES)} companies")
for name, aliases in TARGET_COMPANIES:
    print(f"  {name}: {', '.join(aliases)}")

## 3. Query GDELT GKG (Global Knowledge Graph)

The GKG table extracts themes, organizations, locations, and emotions from news articles. We search for articles mentioning target companies AND infrastructure/investment themes.

**Schema**: `gdelt-bq.gdeltv2.gkg_partitioned`
- `V2Organizations`: semicolon-delimited `Name,Count` pairs
- `V2Themes`: semicolon-delimited `THEME,Count` pairs
- `V2Locations`: location mentions with lat/lng
- `DocumentIdentifier`: article URL
- `SourceCommonName`: source name
- `V2Tone`: tonal score (positive/negative)

**Cost note**: Partitioned by date. Filtering on `_PARTITIONTIME` reduces scanned bytes significantly.

In [ ]:
def query_gkg_by_company(client, company_keywords, start_date, end_date, limit=200):
    """Query GKG for articles mentioning a company + investment/infrastructure themes."""
    
    # Build WHERE clause: company name OR variants
    org_filters = ' OR '.join(
        f"V2Organizations LIKE '%{kw}%'" for kw in company_keywords
    )
    
    # Infrastructure/investment theme keywords
    theme_filters = " OR ".join(
        f"V2Themes LIKE '%{t}%'" for t in [
            'ECON_INVESTMENT', 'INFRASTRUCTURE', 'CONSTRUCTION',
            'FACILITY', 'BUILD', 'ENERGY', 'DATA_CENTER',
            'MANUFACTURING', 'INDUSTRIAL', 'COMPUTER_AND_ELECTRONIC'
        ]
    )
    
    sql = f"""
    SELECT
      DATE, SourceCommonName, DocumentIdentifier,
      V2Organizations, V2Themes, V2Locations,
      V2Tone, Counts
    FROM `gdelt-bq.gdeltv2.gkg_partitioned`
    WHERE _PARTITIONTIME >= TIMESTAMP("{start_date}")
      AND _PARTITIONTIME < TIMESTAMP("{end_date}")
      AND ({org_filters})
      AND ({theme_filters})
    ORDER BY DATE DESC
    LIMIT {limit}
    """
    
    job = client.query(sql)
    df = job.to_dataframe()
    return df, int(job.total_bytes_billed or 0)


# Test: query for Google + investment/infra (last 9 months)
end = datetime.now()
start = end - timedelta(days=270)

df_gkg, billed = query_gkg_by_company(
    client, ['Google', 'Alphabet'],
    start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d'), limit=50
)
print(f"GKG test: {len(df_gkg)} rows, {billed/1e9:.2f} GB billed")
if len(df_gkg):
    display(df_gkg[['DATE', 'SourceCommonName', 'V2Organizations', 'V2Themes']].head(10))

**Results analysis**: The GKG V2Organizations uses numeric IDs after the company name (e.g. `Google,6646`). The count is a document-level identifier offset, not a relevance score. We need to parse the themes to identify actual buildout announcements vs. general company news.

In [ ]:
def parse_gkg_for_buildouts(df, company_name):
    """Filter GKG results to find likely buildout announcements."""
    if df.empty:
        return pd.DataFrame()
    
    # Specific buildout-relevant theme patterns
    BUILDOUT_THEMES = [
        'ECON_INVESTMENT',
        'INFRASTRUCTURE',
        'CONSTRUCTION_NEW_BUILDING',
        'ENERGY',
    ]
    
    records = []
    for _, row in df.iterrows():
        themes = str(row.get('V2Themes', ''))
        # Count buildout-relevant theme mentions
        matches = sum(1 for t in BUILDOUT_THEMES if t in themes)
        if matches >= 1:
            records.append({
                'company': company_name,
                'date': str(row.get('DATE', '')),
                'source': row.get('SourceCommonName', ''),
                'url': row.get('DocumentIdentifier', ''),
                'themes': themes[:500],
                'orgs': str(row.get('V2Organizations', ''))[:300],
                'locations': str(row.get('V2Locations', ''))[:300],
                'tone': row.get('V2Tone', ''),
                'themes_match_count': matches,
            })
    return pd.DataFrame(records)


# Test parse
candidates_gkg = parse_gkg_for_buildouts(df_gkg, 'Google')
print(f"Parsed candidates: {len(candidates_gkg)}")
if len(candidates_gkg):
    display(candidates_gkg[['date', 'source', 'url', 'themes_match_count']].head(10))

## 4. Query GDELT Events Table

The Events table has CAMEO-coded events with Actor1, Actor2, EventCode, and geographic location. Events relevant to data center buildouts include:
- `03` series: Express intent to cooperate (incl. economic cooperation)
- `05` series: Engage in diplomatic cooperation (broad, includes corporate announcements)
- `07` series: Provide aid/economic support
- `10` series: Demand (not relevant)

Note: CAMEO event coding was designed for political events, not corporate investment. It may miss many buildout announcements. GKG approach above is more comprehensive.

In [ ]:
def query_events_for_company(client, company_name, start_date, end_date, limit=200):
    """Query Events table for a company's investment-related events."""
    
    sql = f"""
    SELECT
      SQLDATE, Actor1Name, Actor2Name, ActionGeo_FullName,
      ActionGeo_Lat, ActionGeo_Long, EventCode, EventRootCode,
      GoldsteinScale, AvgTone, NumMentions, SOURCEURL
    FROM `gdelt-bq.gdeltv2.events_partitioned`
    WHERE _PARTITIONTIME >= TIMESTAMP("{start_date}")
      AND _PARTITIONTIME < TIMESTAMP("{end_date}")
      AND Actor1Name LIKE '%{company_name}%'
      AND (EventRootCode = '03' OR EventRootCode = '05' OR EventRootCode = '07')
      AND ActionGeo_FullName IS NOT NULL
    ORDER BY NumMentions DESC
    LIMIT {limit}
    """
    
    job = client.query(sql)
    df = job.to_dataframe()
    return df, int(job.total_bytes_billed or 0)


# Test: events for Google last 9 months
df_events, billed_events = query_events_for_company(
    client, 'Google',
    start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d'), limit=50
)
print(f"Events: {len(df_events)} rows, {billed_events/1e9:.2f} GB billed")
if len(df_events):
    display(df_events[['SQLDATE', 'Actor1Name', 'EventCode', 'ActionGeo_FullName', 'SOURCEURL']].head(10))

## 5. Full Pipeline: Query All Companies

Run GKG queries for all target companies. This scans GKG partitioned data across multiple months. Cost scales with number of companies × date range.

**Cost estimate**: ~2-5 GB per company query scanning 9 months → ~20-50 GB total. Within free tier (1 TB/month).

In [ ]:
from datetime import datetime, timedelta
import time

end = datetime.now()
start = end - timedelta(days=365 * 2)  # Last 2 years

all_candidates = []
total_billed = 0

for company_name, aliases in TARGET_COMPANIES:
    print(f"\n=== {company_name} ===")
    
    df_gkg, billed = query_gkg_by_company(
        client, aliases,
        start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d'),
        limit=500
    )
    total_billed += billed
    print(f"  GKG: {len(df_gkg)} articles, {billed/1e9:.2f} GB")
    
    candidates = parse_gkg_for_buildouts(df_gkg, company_name)
    all_candidates.append(candidates)
    print(f"  -> {len(candidates)} candidate buildouts")
    
    time.sleep(0.5)  # Rate limit

print(f"\n{'='*50}")
print(f"Total billed: {total_billed/1e9:.2f} GB (free tier: 1024 GB/month)")
print(f"Total candidates across all companies: {sum(len(c) for c in all_candidates)}")

## 6. Merge & Deduplicate Candidates

Combine GKG candidates, deduplicate by URL, and cross-reference with existing promises.

In [ ]:
if all_candidates:
    df_all = pd.concat(all_candidates, ignore_index=True)
    print(f"Total raw candidates: {len(df_all)}")
    
    # Dedup by URL (keep highest theme_match_count)
    df_all = df_all.sort_values('themes_match_count', ascending=False)
    df_dedup = df_all.drop_duplicates(subset=['url'], keep='first')
    print(f"After dedup (by URL): {len(df_dedup)}")
    
    # Cross-reference with existing
    if len(df_existing):
        existing_urls = set(df_existing['announcement_url'].dropna().str.lower())
        df_new = df_dedup[~df_dedup['url'].str.lower().isin(existing_urls)].copy()
        print(f"New (not in existing): {len(df_new)}")
    else:
        df_new = df_dedup
    
    display(df_new[['company', 'date', 'source', 'url', 'themes_match_count']].head(20))

## 7. Parse Locations from GKG

The GKG V2Locations field contains unstructured location data (e.g. `VA##USVA#...`, `Boston, Massachusetts, US`). We extract state and city information to match with county demographics.

In [ ]:
def extract_location_info(loc_str):
    """Extract city, state, country from GKG location format.
    
    Format: type#FullName#Country#ADM1#ADM2#lat#lng#featureID
    Example: 2#Columbus, Ohio, United States#US#USOH##39.9612#-82.9988#OH#2343
    """
    if pd.isna(loc_str) or not loc_str:
        return {}
    parts = loc_str.split(';')[0].split('#') if ';' in str(loc_str) else str(loc_str).split('#')
    if len(parts) >= 3:
        full_name = parts[1] if len(parts) > 1 else ''
        # Try to parse "City, State, Country"
        if full_name:
            name_parts = full_name.split(',')
            return {
                'city': name_parts[0].strip() if len(name_parts) > 0 else '',
                'state': name_parts[1].strip() if len(name_parts) > 1 else '',
                'country': name_parts[2].strip() if len(name_parts) > 2 else '',
            }
    return {}


# Apply to new candidates
if 'df_new' in locals() and len(df_new):
    locs = df_new['locations'].apply(extract_location_info)
    df_new = pd.concat([df_new, pd.json_normalize(locs)], axis=1)
    display(df_new[['company', 'date', 'city', 'state', 'country', 'url']].head(10))

## 8. Save & Export

Save the candidate buildout announcements for manual validation.

In [ ]:
if 'df_new' in locals() and len(df_new):
    out_path = 'data/raw/buildout_candidates_bigquery.csv'
    df_new.to_csv(out_path, index=False)
    print(f"Saved {len(df_new)} candidates to {out_path}")
    
    # Summary
    print(f"\nBy company:")
    print(df_new['company'].value_counts().to_string())
    print(f"\nBy year:")
    df_new['year'] = df_new['date'].astype(str).str[:4]
    print(df_new['year'].value_counts().sort_index().to_string())
else:
    print("No candidates found.")

## 9. DVC Tracking (Colab)

On Colab:
```
dvc add data/raw/buildout_candidates_bigquery.csv
dvc push
```

Then locally:
```
dvc pull data/raw/buildout_candidates_bigquery.csv.dvc
```

In [ ]:
# Optional: track with DVC
if IS_COLAB:
    !dvc add data/raw/buildout_candidates_bigquery.csv
    !dvc push
    print("DVC tracking complete")
else:
    print("Local mode: DVC tracking skipped (run on Colab)")

## 10. Validation: Can GDELT Find Known Buildouts?

Check if GDELT captures the existing 34 buildout promises. This measures recall — how many known announcements appear in GDELT.

In [ ]:
# Check GKG for existing buildout URLs
if len(df_existing):
    known_urls = df_existing['announcement_url'].dropna().tolist()
    print(f"Existing announcements with URLs: {len(known_urls)}")
    
    found = 0
    for url in known_urls[:5]:  # Sample first 5
        domain = url.split('/')[2] if '//' in url else url
        sql = f"""
        SELECT COUNT(*) as cnt
        FROM `gdelt-bq.gdeltv2.gkg_partitioned`
        WHERE DocumentIdentifier LIKE '%{domain}%'
          AND DocumentIdentifier LIKE '%{url.split('/')[-1][:30]}%'
        """
        try:
            result = client.query(sql).result()
            cnt = list(result)[0].cnt
            if cnt > 0:
                found += 1
                print(f"  \u2705 Found: {domain}")
            else:
                print(f"  \u274c Not in GDELT: {domain}")
        except Exception as e:
            print(f"  \u26a0 Error checking {domain}: {e}")
    
    print(f"\nRecall: {found}/{min(5, len(known_urls))} known URLs found in GDELT")

## Summary

This notebook:
1. Queries GDELT GKG for articles mentioning target companies with investment/infrastructure themes
2. Parses results into candidate buildout announcements with company, date, location, URL
3. Deduplicates and cross-references with existing promises
4. Saves candidates for manual validation

**Next steps after running**:
- Manually review candidates (check URLs for actual buildout announcements)
- Extract: promised_mw, location, target_date, actual_date
- Add labels (promise_kept) based on news follow-up
- Merge into buildout_promises_expanded.csv
- Retrain ML models with expanded dataset